# Transform Row Data (Silver)

Flattens the raw JSON API response stored in bronze into a clean, tabular silver table — one row per fixture.

* **Source:** `football_col_dev.bronze.api_fixtures` (raw JSON from API-Football, ingested by the AutoLoader notebook)
* **Target:** `football_col_dev.silver.api_fixtures` (MERGE — inserts fixtures not yet present, keyed by `id`)

**How it works**
1. Loads shared imports and the run logger via `%run ../common/01_Setup`.
2. Loads each schema from its own notebook in `../Schemas/` (`03_Response_schema`, `02_Parameters_schema`, `01_Bronze_explode_schema`) via `%run`.
3. Parses the `response` / `parameters` JSON columns with `from_json` + `explode` and flattens them into one row per fixture.
4. Deduplicates to the latest ingest per fixture id (window `row_number` over `_ingested_at`), derives `round` / `number_round` from the API round label (null-safe `try_cast`), uses null-safe `try_cast` for the date, and enforces the explicit 22-column output schema (21 fixture columns incl. `number_round` + `_ingested_at`) with `.to(bronze_explode_schema)`.
5. Merges into silver keyed by `id`: inserts fixtures not yet present, and updates an existing row whenever the derived values differ (NULL-safe `IS DISTINCT FROM` over all columns) — so both newer ingests and transform-logic changes propagate.

**Run:** top to bottom, after the AutoLoader notebook. Every step logs progress, row counts, and inserted rows.


In [0]:
%run ../common/02_Config

In [0]:
%run ../common/01_Setup

In [0]:
%run ../Schemas/03_Response_schema

In [0]:
%run ../Schemas/02_Parameters_schema

In [0]:
%run ../Schemas/01_Bronze_explode_schema

In [0]:
# Run logger from ../common/01_Setup (loaded via %run), tagged to this notebook
logger = get_logger("transform_row_data")

# Configuration comes from ../common/02_Config: BRONZE_TABLE, SILVER_TABLE.
source_table = BRONZE_TABLE
target_table = SILVER_TABLE

logger.info(f"Silver transform run started | source: {source_table} | target: {target_table}")

# Raw bronze rows: one row per landed JSON file, plus ingestion metadata columns
bronze_df = spark.read.table(source_table)
logger.info(f"Loaded {bronze_df.count()} rows from {source_table}")

In [0]:
# Bronze is append-only: the same fixture can arrive in several landed files.
# Rank rows per fixture id by ingestion time and keep only the latest ingest.
latest = Window.partitionBy("id").orderBy(F.desc("_ingested_at"))

bronze_explode_df = (
    bronze_df
        .select(
            # ingestion metadata — ranks duplicates for dedup, and is kept in the output
            # so the silver MERGE can update a row when a newer ingest arrives
            F.col("_ingested_at"),
            F.col("_source_file"),
            # explode the JSON response array into one row per fixture, typed by the DDL schemas from ../Schemas/
            F.explode(F.from_json(F.col("response"), response_schema)).alias("response"),
            F.from_json(F.col("parameters"), parameters_schema).alias("parameters"),
        )
        .withColumns({
            "id": F.col("response.fixture.id"),
            "season": F.col("parameters.season").cast("int"),
            # try_cast: null-safe date parsing (a malformed API value yields NULL instead of failing the run)
            "date": F.date_format(F.expr("try_cast(response.fixture.date AS DATE)"), "dd-MM-yyyy"),
            "league": F.col("response.league.name"),
            "round": F.col("response.league.round"),
            "referee": F.col("response.fixture.referee"),
            "city": F.col("response.fixture.venue.city"),
            "stadium": F.col("response.fixture.venue.name"),
            "home_team": F.col("response.teams.home.name"),
            "away_team": F.col("response.teams.away.name"),
            "home_logo": F.col("response.teams.home.logo"),
            "away_logo": F.col("response.teams.away.logo"),
            "home_goals": F.col("response.goals.home"),
            "away_goals": F.col("response.goals.away"),
            "home_halftime_goals": F.col("response.score.halftime.home"),
            "away_halftime_goals": F.col("response.score.halftime.away"),
            "home_penalty_goals": F.col("response.score.penalty.home"),
            "away_penalty_goals": F.col("response.score.penalty.away"),
            "home_winner": F.col("response.teams.home.winner"),
            "away_winner": F.col("response.teams.away.winner"),
            "league_id": F.col("response.league.id"),
            "status": F.col("response.fixture.status.short"),
            "status_long": F.col("response.fixture.status.long"),
            "venue_id": F.col("response.fixture.venue.id"),
            "home_team_id": F.col("response.teams.home.id"),
            "away_team_id": F.col("response.teams.away.id"),
        })
        .withColumn("is_finished", F.col("status").isin("FT", "AET", "PEN", "AWD", "WO"))
        # keep only the latest ingest per fixture id
        .withColumn("rn", F.row_number().over(latest))
        # split the API round label into phase and number, e.g. 'Apertura - 1' -> ('Apertura', 1)
        # element_at(-1) takes the LAST part, so 'Apertura - Quadrangular - 1' still yields 1
        .withColumn("round_parts", F.split(F.col("round"), "-"))
        .withColumn("round", F.trim(F.col("round_parts").getItem(0)))
        # try_cast: unnumbered rounds (e.g. 'Clausura - Final') become NULL instead of failing
        .withColumn("number_round", F.expr("try_cast(trim(element_at(round_parts, -1)) AS BIGINT)"))
        .withColumn("round", F.when(
            F.col("number_round").isNull(),
            F.concat(F.col("round"), F.lit(" "), F.lit("Final"))
        ).otherwise(F.col("round")))
        .withColumn("number_round", F.when(
            F.col("number_round").isNull(),
            F.lit(0)
        ).otherwise(F.col("number_round")))
        .filter(F.col("rn") == 1)
        .drop("response", "parameters", "rn", "round_parts")
        # enforce the explicit 22-column output schema (21 fixture columns incl. number_round + _ingested_at) from ../Schemas/bronze_explode_schema
        .to(bronze_explode_schema)
)


logger.info(f"Flattened {bronze_explode_df.count()} fixtures (deduplicated to the latest ingest per fixture id)")

In [0]:
# Register the flattened fixtures as a temp view for ad-hoc SQL exploration
bronze_explode_df.createOrReplaceTempView("bronze_explode_df")



In [0]:
%sql
select * from bronze_explode_df


In [0]:
logger.info(f"Merging {bronze_explode_df.count()} fixtures into {target_table} on key id")

# Create the silver table (empty, with the enforced schema) only on the very first run
if not spark.catalog.tableExists(target_table):
    bronze_explode_df.limit(0).write.saveAsTable(target_table)
    logger.info(f"Target table {target_table} did not exist — created with the enforced schema")

silver = DeltaTable.forName(spark, target_table)

# One-time migration: earlier silver versions may lack newer output columns (e.g. number_round).
# Silver is fully derived from bronze, so rebuild it from the current source (pure PySpark, no SQL DDL).
missing_columns = [c for c in bronze_explode_df.columns if c not in silver.toDF().columns]
if missing_columns:
    (
        bronze_explode_df
            .write.mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
    )
    silver = DeltaTable.forName(spark, target_table)
    logger.info(f"Schema migration: rebuilt {target_table} to add columns {missing_columns}")

# Idempotent MERGE keyed on fixture id:
#   * update an existing row whenever the derived values actually differ — this covers both
#     newer ingests (a newer ingest changes _ingested_at) AND transform-logic changes
#     (a code change alone does not bump _ingested_at, so a timestamp-only condition would skip it)
#   * NULL-safe comparison via IS DISTINCT FROM
#   * insert fixtures not yet in silver
row_differs = " OR ".join(
    f"t.{column} IS DISTINCT FROM s.{column}"
    for column in bronze_explode_df.columns
    if column != "id"
)
(
    silver.alias("t")
        .merge(bronze_explode_df.alias("s"), "t.id = s.id")
        .whenMatchedUpdateAll(condition=row_differs)
        .whenNotMatchedInsertAll()
        .execute()
)

# Rows actually written, from the latest Delta commit's operation metrics
# (Delta API history + DataFrame column access — no spark.sql / selectExpr)
metrics = (
    DeltaTable.forName(spark, target_table)
        .history(1)
        .select(
            F.col("operationMetrics").getItem("numTargetRowsInserted").alias("inserted"),
            F.col("operationMetrics").getItem("numTargetRowsUpdated").alias("updated"),
        )
        .first()
)

logger.info(
    f"Silver transform run completed | {target_table}: "
    f"{metrics['inserted']} new fixtures inserted, {metrics['updated']} updated (newer ingest or changed values)"
)

In [0]:
# Preview the silver table (table name comes from 02_Config)
display(spark.table(SILVER_TABLE))